In [3]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import optuna
import optuna.logging

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
)

from utils.utils import (
    connection,
    data_from_ticker,
    data_from_tpulse,
    data_from_macrofactors,
)
import utils.config_ml as config_ml

In [4]:
companies = pd.read_sql("SELECT * FROM companies", connection())
tickers = companies['ticker']

In [5]:
left_date = config_ml.LEFT_DATE
right_date = config_ml.RIGHT_DATE
train_period = config_ml.TRAIN_PERIOD
val_period = config_ml.VAL_PERIOD
test_period = config_ml.TEST_PERIOD
step = config_ml.STEP
models = config_ml.MODELS
n_trials = config_ml.N_TRIALS
metric_optuna = config_ml.METRIC_OPTUNA
top_n_features = config_ml.TOP_N_FEATURES

In [6]:
def pack_all_data_for_ml_models(ticker: str, left_date: str, right_date: str, conn):
    tpulse_data = data_from_tpulse(ticker, left_date, right_date, conn)
    ticker_data = data_from_ticker(ticker, left_date, right_date, conn)
    macrofactor_data = data_from_macrofactors(ticker, left_date, right_date, conn)

    if tpulse_data.empty or ticker_data.empty or macrofactor_data.empty:
        return pd.DataFrame()
    
    data = tpulse_data.merge(macrofactor_data, how='left', on=['dt', 'ticker'])
    data = data.merge(ticker_data, how='left', on=['dt', 'ticker'])
    
    data = data[~data['target'].isnull()].reset_index(drop=True)

    return data